# Reproducing the main results

This notebook checks that the code of this repository produces the numbers in the README. It does
**not** retrain anything: it downloads the LoRA adapters we published on Hugging Face, evaluates
them again on the test set from scratch (no cached results are used), and compares the new
predictions with the ones we saved during our runs.

- **No Hugging Face token is needed.** All datasets, adapters and result tables of the
  [bdatm-project](https://huggingface.co/bdatm-project) organization are public, and this notebook
  never writes to the Hub.
- **GPU:** an NVIDIA L4 or A100 (on Colab: *Runtime > Change runtime type*). The model runs in
  bfloat16, which the free T4 GPU does not support well.
- **Decoding is greedy** (no sampling), so on the same GPU type the predictions should be
  identical to ours, or differ on one or two samples. A different GPU type can change a few more,
  because floating-point results on GPUs depend slightly on the hardware.

The `LEVEL` setting in Section 2 chooses how much to run. The outputs saved in this notebook come
from `LEVEL = "tables"`; to check the code quickly, choose `"main"` before running:

| `LEVEL` | What runs | Time on an L4 |
|---|---|---|
| `"main"` (quick check) | The raster LoRA model of each task, on all 400 test samples | about 35 min |
| `"tables"` (the saved outputs) | Every row of the README tables, including the 512px resolution ablation. Task 2 and Task 3 on all 400 samples; Task 1 on all 400 samples for the raster LoRA model and on the first 100 for the other rows | about 2.5 h |
| `"full"` | Every row of the README tables, including the 512px resolution ablation, on all 400 test samples | about 6.5 h |

These times are for the evaluation. Downloading the three datasets without a token takes about
40 minutes more in total (about 10 minutes each for Task 1 and Task 2, 20 for Task 3, measured on
our connection).

Task 1 is the slow one: a FEN answer is about 40 tokens long and is generated one token at a time.
Using 100 samples is still a fair check, because the comparison is done sample by sample against
our saved predictions for those same samples.

## 1. Setup

On Colab this cell clones the repository and installs the pinned packages of `requirements.txt`
(except `torch` and `torchvision`, which are preinstalled on Colab), and removes Colab's `torchao`,
which is too old for `peft`. To run locally, set `CONFIG["colab"] = False`, install the packages
with `pip install -r requirements.txt` and start Jupyter from the `notebooks/` folder.

In [7]:
import subprocess
from pathlib import Path

CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

if CONFIG["colab"]:
    repo_root = Path(CONFIG["repo_dir"])
    if not repo_root.exists():
        repo_url = f"https://github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"
        print(f"Cloning {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True,
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
    # Colab already has torch and torchvision: install everything else at the pinned versions
    requirements = [line for line in (repo_root / "requirements.txt").read_text().splitlines()
                    if line.strip() and not line.startswith(("#", "torch"))]
    Path("/tmp/requirements_colab.txt").write_text("\n".join(requirements))
    !pip install -q -r /tmp/requirements_colab.txt
    # Colab's preinstalled torchao is too old for peft, which then fails to load the adapters;
    # the task notebooks remove it too
    !pip uninstall -y -q torchao
else:
    repo_root = Path(".").resolve().parent

print(f"Repository: {repo_root}")

Repository: /content/BigDataAndTextMiningProject


In [8]:
import gc
import io
import json
import os
import sys

import pandas as pd
import torch
import transformers
import peft
from datasets import load_dataset
from huggingface_hub import hf_hub_download, snapshot_download
from peft import PeftModel
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

sys.path.insert(0, str(repo_root / "src"))

from eval.evaluation import (
    evaluate_chessboard_model_task_1,
    evaluate_chessboard_model_task_2,
    evaluate_chessboard_model_task_3,
)
from eval.diagnostics import analyze_fen_predictions
from eval.results import load_results_table
from training.finetune import experiment_model_name, experiment_repo_id
from training.reordering import resolve_square_order
from training.resolution import configure_image_resolution
from training.token_reordering import install_token_reordering, verify_token_reordering

print(f"torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}")
assert torch.cuda.is_available(), "This notebook needs a GPU (an L4 or A100)."
print(f"GPU: {torch.cuda.get_device_name(0)}")
if not torch.cuda.is_bf16_supported():
    print("WARNING: this GPU does not support bfloat16 well; results may differ from ours.")

torch 2.11.0+cu130, transformers 5.20.2, peft 0.21.2
GPU: NVIDIA L4


## 2. Settings

`LEVEL` chooses which rows to reproduce (see the table at the top). On Colab it is a drop-down menu
next to the code; elsewhere, edit the value. The outputs saved in this notebook come from
`LEVEL = "tables"`; for a quick check choose `"main"`.

`NUM_WORKERS` sets how many processes download and load the datasets in parallel. It only changes
the loading time, not the results; a split stored as a single file is loaded by one process
whatever its value. The other settings are the ones used in the task notebooks and should not be
changed.

In [9]:
LEVEL = "tables"  # @param ["main", "tables", "full"]
TASK1_SUBSET = 100      # with LEVEL = "tables": test samples for the Task 1 rows other than raster LoRA
NUM_WORKERS = 8  # @param {type:"integer"}

TASKS = ["task1", "task2", "task3"]
MODEL_ID = "Qwen/Qwen3.5-0.8B"
HF_ORG = "bdatm-project"
IMAGE_SIZE = 256        # 256px -> one visual token per chess square
ABLATION_IMAGE_SIZE = 512   # resolution ablation: 2x2 visual tokens per square
TAG = f"tok{IMAGE_SIZE}"
STRATEGIES = ["zigzag", "spiral_center_out", "file_wise", "hilbert"]
N_SHOT = 3              # solved train examples in the prompt of the 3-shot vanilla model

# Main score of each task, as in the README: Task 1 compares the board part of the FEN,
# Tasks 2 and 3 the SAN move taken from the first line of the answer
MAIN_METRIC = {"task1": "board_exact_match", "task2": "em_extracted", "task3": "em_extracted"}
EVAL_FUNCS = {
    "task1": evaluate_chessboard_model_task_1,
    "task2": evaluate_chessboard_model_task_2,
    "task3": evaluate_chessboard_model_task_3,
}
assert LEVEL in ("main", "tables", "full"), f"Unknown LEVEL: {LEVEL}"

## 3. Helper functions

- `load_task_dataset` loads a task's dataset from the Hub the same way the task notebooks do.
  Task 3 has two images per sample, so its files are downloaded first and the second frame
  (`image_t1`) is added to each row.
- `experiment_rows` lists the rows of the README tables for a task: the model name used in the
  results table, the adapter on the Hub (`None` for the vanilla model), the scan order used at
  test time and the image size. For the learned orders, the order is read from
  `learned_order.json` in the adapter's repository. The last row is the 512px raster LoRA model of
  the resolution ablation (repository `qwen-taskN-standard-lora`, named
  `Qwen + LoRA Fine-Tuning (from HF)` in the results table).
- `evaluate_row` loads the base model and the adapter, sets the scan order and runs the task's
  evaluation function of `src/eval/evaluation.py`, with the processor of the row's image size.
- `compare_with_published` downloads our saved per-sample predictions for the same model and
  compares them with the new ones, sample by sample.

In [10]:
def load_task_dataset(task):
    name = f"{HF_ORG}/dataset_{task}"
    if task != "task3":
        return load_dataset(name, num_proc=NUM_WORKERS)
    # load_dataset(name) would download every image one by one and hit the Hub rate limit
    local_dir = snapshot_download(repo_id=name, repo_type="dataset", local_dir=f"./{task}_files",
                                  max_workers=NUM_WORKERS)
    dataset = load_dataset("imagefolder", data_dir=local_dir, num_proc=NUM_WORKERS)
    for split in dataset.keys():
        def add_second_frame(example, split=split):
            path = os.path.join(local_dir, split, example["file_name_t1"])
            example["image_t1"] = Image.open(path).convert("RGB")
            return example
        dataset[split] = dataset[split].map(add_second_frame, num_proc=NUM_WORKERS)
    return dataset


def load_base_model():
    return AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def learned_order(adapter_repo):
    """Scan order used to evaluate a learned-reordering adapter."""
    with open(hf_hub_download(adapter_repo, "learned_order.json")) as f:
        return json.load(f)["order"]


def experiment_rows(task):
    raster = experiment_repo_id(task, "raster", TAG, HF_ORG)
    rows = [
        {"name": "Vanilla Qwen3.5-0.8B (Zero-Shot)", "adapter": None, "order": None},
        {"name": f"Vanilla Qwen3.5-0.8B ({N_SHOT}-Shot)", "adapter": None, "order": None, "few_shot": True},
        {"name": "Qwen + LoRA (Raster - SFT)", "adapter": raster, "order": None},
    ]
    # Training-free reordering: the raster model reads the tokens in another order
    rows += [{"name": f"Qwen + LoRA Raster, read {s} (TF)", "adapter": raster, "order": s} for s in STRATEGIES]
    # Fine-tuned reordering: one adapter per order, tested with that order
    rows += [{"name": f"Qwen + LoRA ({s.capitalize()} - SFT)", "adapter": experiment_repo_id(task, s, TAG, HF_ORG),
              "order": s} for s in STRATEGIES]
    for strategy, label in [("learned", "Learned Reordering"), ("learned-frozen", "Learned Reordering, frozen policy")]:
        repo = experiment_repo_id(task, strategy, TAG, HF_ORG)
        rows.append({"name": f"Qwen + LoRA ({label})", "adapter": repo, "order": learned_order(repo)})
    for row in rows:
        row["name"] = experiment_model_name(row["name"], TAG)
        row["label"] = row["name"]
        row["image_size"] = IMAGE_SIZE
        row["is_main"] = row["adapter"] == raster and row["order"] is None
    # Resolution ablation: the raster LoRA model trained and tested at 512px
    rows.append({
        "name": "Qwen + LoRA Fine-Tuning (from HF)",   # its name in the results table
        "label": "Qwen + LoRA (Raster - SFT) [512px]",
        "adapter": f"{HF_ORG}/qwen-{task}-standard-lora",
        "order": None,
        "image_size": ABLATION_IMAGE_SIZE,
        "is_main": False,
    })
    return rows


def selected_rows(task):
    """The rows to run for this LEVEL, each with its number of test samples."""
    rows = [r for r in experiment_rows(task) if LEVEL != "main" or r["is_main"]]
    for row in rows:
        subset = LEVEL == "tables" and task == "task1" and not row["is_main"]
        row["n_samples"] = TASK1_SUBSET if subset else None   # None = all test samples
    return rows


def evaluate_row(task, row, dataset, processors):
    model = load_base_model()
    if row["adapter"] is not None:
        model = PeftModel.from_pretrained(model, row["adapter"])
    test = dataset["test"]
    if row["n_samples"] is not None:
        test = test.select(range(row["n_samples"]))
    few_shot = [dataset["train"][i] for i in range(N_SHOT)] if row.get("few_shot") else None
    reorderer = install_token_reordering(model)
    with reorderer.order(resolve_square_order(row["order"])):
        results_df, _ = EVAL_FUNCS[task](
            model=model, processor=processors[row["image_size"]], dataset_split=test, model_name=row["name"],
            few_shot_examples=few_shot,
        )
    del model, reorderer
    free_gpu()
    return results_df


def compare_with_published(task, row, results_df, published_table):
    repo = f"{HF_ORG}/evaluation-results-{task}"
    slug = row["name"].lower().replace(" ", "_")
    published = pd.read_csv(hf_hub_download(repo, f"per_sample/{slug}.csv", repo_type="dataset"))
    published = published.set_index("sample_id").loc[results_df["sample_id"]].reset_index()
    if task == "task1" and MAIN_METRIC[task] not in published.columns:
        published = analyze_fen_predictions(published)   # older saved results lack the board columns
    # write the new results to CSV and back, so both sides are parsed the same way
    new = pd.read_csv(io.StringIO(results_df.to_csv(index=False)))
    same = new["predicted"].fillna("").astype(str).values == published["predicted"].fillna("").astype(str).values
    metric = MAIN_METRIC[task]
    readme_value = published_table.loc[published_table["model_name"] == row["name"], metric].iloc[0]
    return {
        "task": task,
        "model": row["label"],
        "test samples": len(new),
        "ours, same samples (%)": round(100 * published[metric].mean(), 2),
        "reproduced (%)": round(100 * new[metric].mean(), 2),
        "identical predictions": f"{same.sum()}/{len(same)}",
        "ours, 400 samples (README, %)": round(100 * readme_value, 2),
    }

## 4. Processors and token reordering check

Two processors are prepared: one resizes the boards to 256px (all the main experiments) and one to
512px (the resolution ablation). Each prints the token grid it gives: 8x8 tokens at 256px (one per
square) and 16x16 at 512px (2x2 per square).

The scan orders are applied by patching the model's code at run time
(`src/training/token_reordering.py`). `verify_token_reordering` checks that the patch works with
the installed `transformers` version: with no order set the output equals the unpatched model,
the token embeddings and their positions move together, and setting an order changes the output.

In [11]:
processors = {}
for size in (IMAGE_SIZE, ABLATION_IMAGE_SIZE):
    processors[size] = AutoProcessor.from_pretrained(MODEL_ID)
    configure_image_resolution(processors[size], size)
processor = processors[IMAGE_SIZE]

check_dataset = load_dataset(f"{HF_ORG}/dataset_task1", split="test[:1]")
check_model = load_base_model()
verify_token_reordering(check_model, processor, check_dataset[0]["image"], resolve_square_order("zigzag"))
del check_model
free_gpu()

[resolution] processor set to 256px: {'image_size': 256, 'patch_grid': (16, 16), 'llm_token_grid': (8, 8), 'tokens_per_square': 1}
[resolution] processor set to 512px: {'image_size': 512, 'patch_grid': (32, 32), 'llm_token_grid': (16, 16), 'tokens_per_square': 4}


Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[token reordering] OK: {'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.0625}


## 5. Evaluate and compare

For each task: load the dataset and our published results table, then evaluate every selected row
and compare it with our saved predictions. Every row prints a progress bar; Task 1 rows take
about 20-25 minutes each on all 400 samples, Task 2 and Task 3 rows about 2 minutes. The 512px
rows are slower, because each image gives 4 times as many visual tokens.

In the comparison table:

- **ours, same samples**: the score of our saved predictions on the samples evaluated here;
- **reproduced**: the score of the new predictions on the same samples;
- **identical predictions**: how many new predictions are exactly the same string as ours;
- **ours, 400 samples (README)**: the number reported in the README.

### Task 1

In [12]:
TASK = "task1"
dataset = load_task_dataset(TASK)
published_table = load_results_table(f"{HF_ORG}/evaluation-results-{TASK}")

comparison_task1 = []
for row in selected_rows(TASK):
    print(f"\n=== {row['label']} ({row['n_samples'] or len(dataset['test'])} test samples) ===")
    results_df = evaluate_row(TASK, row, dataset, processors)
    comparison_task1.append(compare_with_published(TASK, row, results_df, published_table))
    print(comparison_task1[-1])

display(pd.DataFrame(comparison_task1))
del dataset

Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]


=== Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256] (100 test samples) ===


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

...

Evaluating Qwen + LoRA Fine-Tuning (from HF): 100%|██████████| 100/100 [04:59<00:00,  2.99s/it]



Evaluation completed for Qwen + LoRA Fine-Tuning (from HF)! Results saved to task1_qwen_+_lora_fine-tuning_(from_hf)_results.csv.


(…)n_%2B_lora_fine-tuning_%28from_hf%29.csv:   0%|          | 0.00/82.2k [00:00<?, ?B/s]

{'task': 'task1', 'model': 'Qwen + LoRA (Raster - SFT) [512px]', 'test samples': 100, 'ours, same samples (%)': np.float64(100.0), 'reproduced (%)': np.float64(100.0), 'identical predictions': '84/100', 'ours, 400 samples (README, %)': np.float64(98.75)}


,task,model,test samples,"ours, same samples (%)",reproduced (%),identical predictions,"ours, 400 samples (README, %)"
0,task1,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],100,0.0,0.0,90/100,0.00
1,task1,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],100,0.0,0.0,70/100,0.00
2,task1,Qwen + LoRA (Raster - SFT) [tok256],400,97.0,97.0,342/400,97.00
3,task1,"Qwen + LoRA Raster, read zigzag (TF) [tok256]",100,0.0,0.0,59/100,0.00
4,task1,"Qwen + LoRA Raster, read spiral_center_out (TF...",100,0.0,0.0,62/100,0.00
5,task1,"Qwen + LoRA Raster, read file_wise (TF) [tok256]",100,0.0,0.0,47/100,0.00
6,task1,"Qwen + LoRA Raster, read hilbert (TF) [tok256]",100,0.0,0.0,63/100,0.00
7,task1,Qwen + LoRA (Zigzag - SFT) [tok256],100,97.0,97.0,83/100,97.25
8,task1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],100,99.0,99.0,82/100,98.50
9,task1,Qwen + LoRA (File_wise - SFT) [tok256],100,95.0,95.0,78/100,96.75


### Task 2

In [13]:
TASK = "task2"
dataset = load_task_dataset(TASK)
published_table = load_results_table(f"{HF_ORG}/evaluation-results-{TASK}")

comparison_task2 = []
for row in selected_rows(TASK):
    print(f"\n=== {row['label']} ({row['n_samples'] or len(dataset['test'])} test samples) ===")
    results_df = evaluate_row(TASK, row, dataset, processors)
    comparison_task2.append(compare_with_published(TASK, row, results_df, published_table))
    print(comparison_task2[-1])

display(pd.DataFrame(comparison_task2))
del dataset

Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

metadata.jsonl:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

metadata.jsonl:   0%|          | 0.00/213k [00:00<?, ?B/s]

...

Evaluating Qwen + LoRA Fine-Tuning (from HF) (Task 2): 100%|██████████| 400/400 [02:14<00:00,  2.97it/s]



Evaluation completed for Qwen + LoRA Fine-Tuning (from HF)! Results saved to task2_qwen_+_lora_fine-tuning_(from_hf)_results.csv.


(…)n_%2B_lora_fine-tuning_%28from_hf%29.csv:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

{'task': 'task2', 'model': 'Qwen + LoRA (Raster - SFT) [512px]', 'test samples': 400, 'ours, same samples (%)': np.float64(62.0), 'reproduced (%)': np.float64(62.0), 'identical predictions': '400/400', 'ours, 400 samples (README, %)': np.float64(62.0)}


,task,model,test samples,"ours, same samples (%)",reproduced (%),identical predictions,"ours, 400 samples (README, %)"
0,task2,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],400,0.00,0.00,400/400,0.00
1,task2,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],400,0.75,0.75,400/400,0.75
2,task2,Qwen + LoRA (Raster - SFT) [tok256],400,60.25,60.25,400/400,60.25
3,task2,"Qwen + LoRA Raster, read zigzag (TF) [tok256]",400,52.75,52.75,400/400,52.75
4,task2,"Qwen + LoRA Raster, read spiral_center_out (TF...",400,14.50,14.50,400/400,14.50
5,task2,"Qwen + LoRA Raster, read file_wise (TF) [tok256]",400,9.75,9.75,400/400,9.75
6,task2,"Qwen + LoRA Raster, read hilbert (TF) [tok256]",400,29.00,29.00,400/400,29.00
7,task2,Qwen + LoRA (Zigzag - SFT) [tok256],400,53.50,54.00,395/400,53.50
8,task2,Qwen + LoRA (Spiral_center_out - SFT) [tok256],400,54.00,53.75,399/400,54.00
9,task2,Qwen + LoRA (File_wise - SFT) [tok256],400,54.75,54.75,397/400,54.75


### Task 3

In [14]:
TASK = "task3"
dataset = load_task_dataset(TASK)
published_table = load_results_table(f"{HF_ORG}/evaluation-results-{TASK}")

comparison_task3 = []
for row in selected_rows(TASK):
    print(f"\n=== {row['label']} ({row['n_samples'] or len(dataset['test'])} test samples) ===")
    results_df = evaluate_row(TASK, row, dataset, processors)
    comparison_task3.append(compare_with_published(TASK, row, results_df, published_table))
    print(comparison_task3[-1])

display(pd.DataFrame(comparison_task3))
del dataset

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8004 files:   0%|          | 0/8004 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/6401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

...

Evaluating Qwen + LoRA Fine-Tuning (from HF) on Task 3: 100%|██████████| 400/400 [02:18<00:00,  2.89it/s]



Evaluation completed for Qwen + LoRA Fine-Tuning (from HF) on Task 3! Results saved to task3_qwen_+_lora_fine-tuning_(from_hf)_results.csv.


(…)n_%2B_lora_fine-tuning_%28from_hf%29.csv:   0%|          | 0.00/19.8k [00:00<?, ?B/s]

{'task': 'task3', 'model': 'Qwen + LoRA (Raster - SFT) [512px]', 'test samples': 400, 'ours, same samples (%)': np.float64(87.5), 'reproduced (%)': np.float64(87.5), 'identical predictions': '400/400', 'ours, 400 samples (README, %)': np.float64(87.5)}


,task,model,test samples,"ours, same samples (%)",reproduced (%),identical predictions,"ours, 400 samples (README, %)"
0,task3,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],400,0.25,0.25,400/400,0.25
1,task3,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],400,0.00,0.00,400/400,0.00
2,task3,Qwen + LoRA (Raster - SFT) [tok256],400,79.75,79.75,400/400,79.75
3,task3,"Qwen + LoRA Raster, read zigzag (TF) [tok256]",400,42.25,42.25,400/400,42.25
4,task3,"Qwen + LoRA Raster, read spiral_center_out (TF...",400,5.75,5.75,400/400,5.75
5,task3,"Qwen + LoRA Raster, read file_wise (TF) [tok256]",400,4.00,4.00,400/400,4.00
6,task3,"Qwen + LoRA Raster, read hilbert (TF) [tok256]",400,12.75,12.75,400/400,12.75
7,task3,Qwen + LoRA (Zigzag - SFT) [tok256],400,77.50,77.50,400/400,77.50
8,task3,Qwen + LoRA (Spiral_center_out - SFT) [tok256],400,70.75,70.75,400/400,70.75
9,task3,Qwen + LoRA (File_wise - SFT) [tok256],400,80.50,80.50,400/400,80.50


## 6. Summary

All the reproduced rows together. The scores should match ours and nearly every prediction should
be identical; on a different GPU type a few predictions can change (see the top of the notebook).

In [15]:
summary = pd.DataFrame(comparison_task1 + comparison_task2 + comparison_task3)
summary["difference (points)"] = (summary["reproduced (%)"] - summary["ours, same samples (%)"]).round(2)
print(f"LEVEL = {LEVEL!r}, GPU: {torch.cuda.get_device_name(0)}")
display(summary)

LEVEL = 'tables', GPU: NVIDIA L4


,task,model,test samples,"ours, same samples (%)",reproduced (%),identical predictions,"ours, 400 samples (README, %)",difference (points)
0,task1,Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256],100,0.00,0.00,90/100,0.00,0.00
1,task1,Vanilla Qwen3.5-0.8B (3-Shot) [tok256],100,0.00,0.00,70/100,0.00,0.00
2,task1,Qwen + LoRA (Raster - SFT) [tok256],400,97.00,97.00,342/400,97.00,0.00
3,task1,"Qwen + LoRA Raster, read zigzag (TF) [tok256]",100,0.00,0.00,59/100,0.00,0.00
4,task1,"Qwen + LoRA Raster, read spiral_center_out (TF...",100,0.00,0.00,62/100,0.00,0.00
5,task1,"Qwen + LoRA Raster, read file_wise (TF) [tok256]",100,0.00,0.00,47/100,0.00,0.00
6,task1,"Qwen + LoRA Raster, read hilbert (TF) [tok256]",100,0.00,0.00,63/100,0.00,0.00
7,task1,Qwen + LoRA (Zigzag - SFT) [tok256],100,97.00,97.00,83/100,97.25,0.00
8,task1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],100,99.00,99.00,82/100,98.50,0.00
9,task1,Qwen + LoRA (File_wise - SFT) [tok256],100,95.00,95.00,78/100,96.75,0.00
